In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('train.csv')
df.head()

,row_id,previous_reply,target_reply,level_1_class,toxicity_type
0,ef2a32c7af5a0a660e0c,I shared photos from today's event.,"In this video, your body draws more attention ...",Toxic,Sexual Harassment / Objectification
1,c7a506deea76fe585973,I see no reason to change my opinion.,"Hearing this statement, it seems imagination d...",Toxic,Mockery / Ridicule
2,5ff6cb853fa2d2809023,"Someone in the discussion said, There is no ne...",Not everyone is the same; people should be tre...,Non-toxic,NaN
3,577f23b0dbafdca5c219,What's the update on today's work?,"So far, More details can be given later.",Non-toxic,NaN
4,ca4f30bb8940a17405a0,I only disagreed politely.,"I'm warning you, if you ask this question agai...",Toxic,Threat


In [3]:
df.drop(columns=['row_id'], inplace=True)

In [4]:
df['toxicity_type'].value_counts()

toxicity_type
Insult                                 1923
Threat                                  902
Mockery / Ridicule                      845
Hate Speech                             808
Incitement / Encouragement of Harm      800
Identity Attack                         800
Dehumanization                          796
Profanity                               779
Sexual Harassment / Objectification     763
Toxic Endorsement / Agreement           621
Harassment                              555
Name: count, dtype: int64

In [5]:
df_test = pd.read_csv('test.csv')
df_test.drop(columns=['Source', 'context', 'reply', 'level_1_class'], inplace=True)
df_test.rename(columns={'context_en': 'context', 'reply_en': 'reply'}, inplace=True)
df_test.head()

,toxicity_type,context,reply
0,NaN,HSC exam will be conducted on time in the enti...,right decision
1,NaN,HSC exam will be conducted on time in the enti...,You are absolutely right
2,Insult,HSC exam will be conducted on time in the enti...,ওরা আস্থা বেয়াদব কোন পদার্থের চলবে না ওদের কথ...
3,Mockery / Ridicule,Why was the test not delayed? Education Minist...,What is the name of the education minister who...
4,NaN,Why was the test not delayed? Education Minist...,What was asked and what was the answer?


In [6]:
df.rename(columns={'previous_reply': 'context', 'target_reply': 'reply'}, inplace=True)
df.head()

,context,reply,level_1_class,toxicity_type
0,I shared photos from today's event.,"In this video, your body draws more attention ...",Toxic,Sexual Harassment / Objectification
1,I see no reason to change my opinion.,"Hearing this statement, it seems imagination d...",Toxic,Mockery / Ridicule
2,"Someone in the discussion said, There is no ne...",Not everyone is the same; people should be tre...,Non-toxic,NaN
3,What's the update on today's work?,"So far, More details can be given later.",Non-toxic,NaN
4,I only disagreed politely.,"I'm warning you, if you ask this question agai...",Toxic,Threat


In [7]:
df['level_1_class'].value_counts()

level_1_class
Non-toxic    9861
Toxic        9592
Name: count, dtype: int64

In [8]:
df.loc[df['level_1_class'] == 'Non-toxic', 'toxicity_type'] = 'None'    ## Setting every Non-toxic row type to NaN

In [9]:
df.shape

(19453, 4)

In [10]:
df['toxicity_type'].value_counts()

toxicity_type
None                                   9861
Insult                                 1923
Threat                                  902
Mockery / Ridicule                      845
Hate Speech                             808
Incitement / Encouragement of Harm      800
Identity Attack                         800
Dehumanization                          796
Profanity                               779
Sexual Harassment / Objectification     763
Toxic Endorsement / Agreement           621
Harassment                              555
Name: count, dtype: int64

In [11]:
df.drop_duplicates(keep='first', inplace=True)  ## Dropping duplicate rows
df.shape

(19431, 4)

In [12]:
from sklearn.model_selection import train_test_split

X = df[['context', 'reply']]
y = df['toxicity_type']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_val.shape, y_train.shape, y_val.shape

((15544, 2), (3887, 2), (15544,), (3887,))

In [ ]:
X_train_context = X_train['context'].values    ## Context
X_train_target = X_train['reply'].values       ## reply

X_val_context = X_val['context'].values
X_val_target = X_val['reply'].values

# X_test_context = df_test['context'].values
# X_test_target = df_test['reply'].values

In [18]:
X_train_context.shape, X_train_target.shape, X_val_context.shape, X_val_target.shape

((15544,), (15544,), (3887,), (3887,))

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

context_vectorizer = TfidfVectorizer(
  ngram_range=(1, 2),
  min_df=2,
  sublinear_tf=True   ## apply smoothing to term frequency
)

target_vectorizer = TfidfVectorizer(
  ngram_range=(1, 2),
  min_df=2,
  sublinear_tf=True
)

X_train_context_tfidf = context_vectorizer.fit_transform(X_train_context)
X_train_target_tfidf  = target_vectorizer.fit_transform(X_train_target)

X_val_context_tfidf = context_vectorizer.transform(X_val_context)
X_val_target_tfidf  = target_vectorizer.transform(X_val_target)

# X_test_context_tfidf = context_vectorizer.transform(X_test_context)
# X_test_target_tfidf  = target_vectorizer.transform(X_test_target)

In [22]:
display(list(context_vectorizer.get_feature_names_out()))

['able',
 'able to',
 'about',
 'about ac',
 'about bicycle',
 'about bike',
 'about blog',
 'about board',
 'about booking',
 'about borrowing',
 'about buying',
 'about car',
 'about carrom',
 'about chess',
 'about coding',
 'about company',
 'about conference',
 'about cooking',
 'about credit',
 'about cricket',
 'about cryptocurrency',
 'about debate',
 'about drawing',
 'about eid',
 'about enrolling',
 'about environmental',
 'about event',
 'about facebook',
 'about finding',
 'about flight',
 'about football',
 'about freelance',
 'about freelancing',
 'about getting',
 'about going',
 'about gre',
 'about gym',
 'about handmade',
 'about health',
 'about her',
 'about hospital',
 'about house',
 'about ielts',
 'about internship',
 'about it',
 'about job',
 'about joining',
 'about laptop',
 'about learning',
 'about load',
 'about making',
 'about market',
 'about me',
 'about meeting',
 'about missing',
 'about morning',
 'about mountain',
 'about movie',
 'about my',
 'a

In [17]:
context_vectorizer.get_feature_names_out().shape, target_vectorizer.get_feature_names_out().shape

((5404,), (7664,))

In [26]:
X_train_context_tfidf.shape, X_train_target_tfidf.shape, X_val_context_tfidf.shape, X_val_target_tfidf.shape

((15544, 5404), (15544, 7664), (3887, 5404), (3887, 7664))

In [29]:
context_vectorizer.transform(["Someone in the discussion said, the people in that group should be grabbed and beaten."])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 27 stored elements and shape (1, 5404)>

In [15]:
from numpy import hstack, vstack

X_train = hstack([
  X_train_context_tfidf,
  X_train_target_tfidf
])

X_val = hstack([
  X_val_context_tfidf,
  X_val_target_tfidf
])

# X_test = hstack([
#   X_test_context_tfidf,
#   X_test_target_tfidf
# ])
print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
# print("Test shape:", X_test.shape)

Train shape: (2,)
Validation shape: (2,)


In [16]:
X_train_context_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 256549 stored elements and shape (15544, 5404)>

In [40]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

model.fit(X_train, y_train)

ValueError: setting an array element with a sequence.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

val_pred = model.predict(X_val)

print("\n===== VALIDATION RESULTS =====")
print(classification_report(y_val, val_pred))